In [ ]:
!pip install highway-env stable-baselines3 gymnasium renderlab
!apt-get install -y xvfb ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 33.0 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
import gymnasium as gym
import highway_env
import numpy as np
import imageio
import matplotlib.pyplot as plt
from IPython.display import Image, display
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

In [ ]:
env = gym.make("highway-v0", render_mode="rgb_array")
obs, info = env.reset()

print("Obs shape:", obs.shape)
print("Action space:", env.action_space)
env.close()

In [ ]:
env = gym.make("highway-v0", render_mode="rgb_array")
obs, _ = env.reset()

frames = []
for _ in range(80):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    frames.append(env.render())
    if done or truncated:
        obs, _ = env.reset()

env.close()
imageio.mimsave("random_agent.gif", frames, fps=15)
display(Image("random_agent.gif"))
print("Random agent rendered")

In [ ]:
# Curriculum Stage 1: Dense passive traffic, no adversaries
stage1_config = {
    "observation": {"type": "Kinematics"},
    "action": {"type": "DiscreteMetaAction"},
    "lanes_count": 4,
    "vehicles_count": 30,
    "duration": 40,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "collision_reward": -2,
    "high_speed_reward": 0.4,
    "right_lane_reward": 0.1,
    "lane_change_reward": 0.0,
    "reward_speed_range": [20, 30],
    "normalize_reward": True,
}

def make_stage1_env():
    env = gym.make("highway-v0", render_mode="rgb_array")
    env.unwrapped.configure(stage1_config)
    return env

print("Stage 1 config ready")

In [ ]:
# Curriculum Stage 2: Aggressive drivers — tailgate, cut off, higher density
stage2_config = {
    **stage1_config,
    "vehicles_count": 40,
    "other_vehicles_type": "highway_env.vehicle.behavior.AggressiveVehicle",
    "collision_reward": -3,
}

def make_stage2_env():
    env = gym.make("highway-v0", render_mode="rgb_array")
    env.unwrapped.configure(stage2_config)
    return env

print("✅ Stage 2 config ready")

In [ ]:
from highway_env.vehicle.behavior import IDMVehicle

class PursuerVehicle(IDMVehicle):
    """
    A vehicle that actively pursues the ego agent —
    matches its lane and accelerates to close distance.
    """
    PURSUIT_GAIN = 1.5  # how aggressively it chases

    def act(self, action=None):
        ego = self.road.vehicles[0]  # ego is always index 0

        # Match ego lane
        if self.lane_index[2] < ego.lane_index[2]:
            action = "LANE_RIGHT"
        elif self.lane_index[2] > ego.lane_index[2]:
            action = "LANE_LEFT"
        else:
            # Same lane — floor it
            action = "FASTER"

        super().act(action)

print("✅ PursuerVehicle defined")

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class RewardLogger(BaseCallback):
    def __init__(self):
        super().__init__()
        self.rewards = []

    def _on_step(self):
        if "episode" in self.locals.get("infos", [{}])[0]:
            r = self.locals["infos"][0]["episode"]["r"]
            self.rewards.append(r)
        return True

In [ ]:
import highway_env.vehicle.behavior as behavior
behavior.PursuerVehicle = PursuerVehicle  # register it

stage3_config = {
    **stage1_config,
    "vehicles_count": 30,
    "other_vehicles_type": "highway_env.vehicle.behavior.PursuerVehicle",
    "collision_reward": -5,
}

def make_stage3_env():
    env = gym.make("highway-v0", render_mode="rgb_array")
    env.unwrapped.configure(stage3_config)
    return env

print("✅ Stage 3 config ready")

In [ ]:
def render_agent(model, make_env_fn, filename="agent.gif", episodes=3):
    env = make_env_fn()
    frames = []
    rewards = []

    for ep in range(episodes):
        obs, _ = env.reset()
        ep_reward = 0
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
            frames.append(env.render())
            ep_reward += reward
            done = done or truncated
        rewards.append(ep_reward)
        print(f"Episode {ep+1} reward: {ep_reward:.2f}")

    env.close()
    imageio.mimsave(filename, frames, fps=15)
    display(Image(filename))
    print(f"✅ Avg reward: {np.mean(rewards):.2f}")
    survival_rate = sum(1 for r in rewards if r > 0) / len(rewards)
    print(f"Survival rate: {survival_rate*100:.1f}%")



In [ ]:
def train_stage(make_env_fn, stage_name, timesteps=150_000, model=None):
    """
    Train PPO on a given stage config.
    If model is passed, fine-tunes from existing weights (curriculum transfer).
    """
    vec_env = make_vec_env(make_env_fn, n_envs=2)
    logger = RewardLogger()

    checkpoint_cb = CheckpointCallback(
        save_freq=25_000,
        save_path=f"./{stage_name}_checkpoints/",
        name_prefix=f"avert_{stage_name}"
    )

    if model is None:
        model = PPO(
            "MlpPolicy",
            vec_env,
            verbose=1,
            n_steps=256,
            batch_size=64,
            n_epochs=10,
            learning_rate=3e-4,
            gamma=0.99,
        )
    else:
        model.set_env(vec_env)

    model.learn(total_timesteps=timesteps, callback=[checkpoint_cb, logger], reset_num_timesteps=False)
    model.save(f"avert_{stage_name}_final")
    print(f"✅ {stage_name} training complete — model saved")
    return model, logger


In [ ]:
# Stage 1 — train from scratch
print("=== STAGE 1: Passive Traffic ===")
model, logger1 = train_stage(make_stage1_env, "stage1", timesteps=150_000)

# Stage 2 — fine-tune from stage 1
print("\n=== STAGE 2: Aggressive Traffic ===")
model, logger2 = train_stage(make_stage2_env, "stage2", timesteps=150_000, model=model)

# Stage 3 — fine-tune from stage 2
print("\n=== STAGE 3: Active Pursuers ===")
model, logger3 = train_stage(make_stage3_env, "stage3", timesteps=150_000, model=model)

In [ ]:
print("=== BASELINE: Training from scratch on Stage 3 ===")
model_scratch, _ = train_stage(make_stage3_env, "stage3_scratch", timesteps=150_000, model=None)

print("\n=== Curriculum Agent on Stage 3 ===")
render_agent(model, make_stage3_env, filename="curriculum_agent.gif")

print("\n=== Scratch Agent on Stage 3 ===")
render_agent(model_scratch, make_stage3_env, filename="scratch_agent.gif")

In [ ]:
# Load best model and transfer to roundabout env
roundabout_config = {
    "observation": {"type": "Kinematics"},
    "action": {"type": "DiscreteMetaAction"},
    "collision_reward": -3,
    "high_speed_reward": 0.4,
    "normalize_reward": True,
}

def make_roundabout_env():
    env = gym.make("roundabout-v0", render_mode="rgb_array")
    env.unwrapped.configure(roundabout_config)
    return env

print("=== Transfer: Fine-tuning on Roundabout ===")
model_transfer = PPO.load("avert_stage3_final")
model_transfer, _ = train_stage(make_roundabout_env, "roundabout_transfer", timesteps=75_000, model=model_transfer)

print("\n=== Rendering Transfer Agent ===")
render_agent(model_transfer, make_roundabout_env, filename="avert_transfer_roundabout.gif")

In [ ]:
def plot_rewards(stage_rewards: dict):
    plt.figure(figsize=(12, 4))
    colors = {"Stage 1": "steelblue", "Stage 2": "darkorange", "Stage 3": "crimson"}
    offset = 0
    for stage, rewards in stage_rewards.items():
        x = range(offset, offset + len(rewards))
        plt.plot(x, rewards, label=stage, color=colors[stage], alpha=0.6)
        plt.axvline(x=offset, linestyle="--", color=colors[stage], alpha=0.3)
        offset += len(rewards)
    plt.xlabel("Episode")
    plt.ylabel("Episode Reward")
    plt.title("AVERT — Curriculum Training Reward Curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig("avert_reward_curves.png", dpi=150)
    plt.show()

In [ ]:
stage_rewards = {
    "Stage 1": logger1.rewards,
    "Stage 2": logger2.rewards,
    "Stage 3": logger3.rewards,
}
plot_rewards(stage_rewards)